In [0]:
import sys

sys.path.insert(0, "../lib")

import utils
import ingestors

# tablename = 'clientes'
# idfield = "idCliente"
# idfield_old = "idCliente"

tablename = dbutils.widgets.get("tablename")
idfield = dbutils.widgets.get("id_field")
idfield_old = dbutils.widgets.get("id_field_old")

catalog = "silver"
schemaname = "upsell"

In [0]:
## campos do CDF de bronze
#%sql
#select * from table_changes('bronze.upsell.clientes', 0,1)

In [0]:
remove_checkpoint = False

if not utils.table_exists(spark, "silver", "upsell", tablename):
    
    print("Criando a tabela", tablename)
    query = utils.import_query(f"{tablename}.sql")
    (spark.sql(query)
          .write
          .format("delta")
          .mode("overwrite") # create or replace do SQL
          .option("overwriteSchema", "true")
          .saveAsTable(f"silver.upsell.{tablename}"))
    
    remove_checkpoint = True

In [0]:
print("Iniciando CDF...")

ingest = ingestors.IngestorCDF(spark=spark,
                               catalog=catalog,
                               schemaname=schemaname,
                               tablename=tablename,
                               id_field=idfield,
                               idfield_old=idfield_old)

if remove_checkpoint:
    dbutils.fs.rm(ingest.checkpoint_location, True)

stream = ingest.execute()
print("Ok.")

In [0]:
## confirmação de dados
# %sql

# SELECT COUNT(*) FROM bronze.upsell.clientes

# UNION ALL

# SELECT COUNT(*) FROM silver.upsell.clientes